# 05 — Figure review and presentation readiness

Use this notebook after `04_results.ipynb`. It reviews the saved tables and figures, regenerates selected chart types through `src.visualize`, and provides a final interpretation checklist for a paper or poster.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import Image, display

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)

from src.paths import ProjectPaths
from src.visualize import plot_bottleneck, plot_sensitivity_poa

paths = ProjectPaths.discover()
assert (paths.tables / 'table2_bottleneck_analysis.csv').exists(), 'Run 04_results first.'
assert (paths.tables / 'table5_sensitivity_analysis.csv').exists(), 'Run 04_results first.'

## Regenerate selected figures from saved tables

This is useful when adjusting a figure for a paper or poster. The underlying calculations are unchanged; the visualization functions receive saved analysis results.

In [ ]:
bottleneck_table = pd.read_csv(paths.tables / 'table2_bottleneck_analysis.csv')
sensitivity_table = pd.read_csv(paths.tables / 'table5_sensitivity_analysis.csv')

# The plotting API uses route indices internally, so retain the table alongside this rendered figure.
from src.config import ROUTES
bottlenecks = {ROUTES.index(row.route): row.marginal_welfare_gain for row in bottleneck_table.itertuples()}
fig_bottlenecks = plot_bottleneck(bottlenecks)
fig_sensitivity = plot_sensitivity_poa(sensitivity_table.to_dict('records'))

## Review generated assets

The official pipeline writes presentation-ready files to `output/`. Seeing the rendered image here catches clipped labels, unreadable legends, and accidental inclusion of the out-of-scope `EXTERNAL` route.

In [ ]:
figures = sorted(paths.figures.glob('*.png'))
pd.DataFrame({
    'figure': [path.name for path in figures],
    'size_kb': [round(path.stat().st_size / 1024, 1) for path in figures],
})

In [ ]:
for path in figures:
    print(path.name)
    display(Image(filename=str(path)))

## Presentation checklist

- Every axis names its quantity and unit.
- Legends are readable at poster scale.
- Route labels use the canonical names in `src/config.py`.
- `EXTERNAL` is not shown in the seven-route allocation comparison.
- The figure caption states the data period, model scenario, and key assumption when relevant.
- A reader can trace every plotted value to a CSV in `output/tables/`.
- Maps are reviewed for route-geometry accuracy before publication.

In [ ]:
maps = sorted(paths.maps.iterdir()) if paths.maps.exists() else []
pd.DataFrame({'map_asset': [path.name for path in maps]})